In [1]:
import pandas as pd
import numpy as np

ball = pd.read_csv("../data/ball_by_ball.csv")
matches = pd.read_csv("../data/matches.csv")
competitions = pd.read_csv("../data/competitions.csv")

In [2]:
t20_comp_ids = competitions.loc[
    competitions["match_format"] == "T20", "comp_id"
].unique()

t20_matches = matches[
    matches["comp_id"].isin(t20_comp_ids)
].copy()

t20_ball = ball[
    ball["match_id"].isin(t20_matches["match_id"])
].copy()

In [3]:
# Add match and competition information to ball-by-ball data

batting_data = t20_ball.merge(
    t20_matches[
        ["match_id", "comp_id", "match_date"]
    ],
    on="match_id",
    how="left"
)

batting_data = batting_data.merge(
    competitions[
        ["comp_id", "comp_name", "season"]
    ],
    on="comp_id",
    how="left"
)

print("Batting dataset shape:", batting_data.shape)

display(
    batting_data[
        ["match_id", "match_date", "comp_name",
         "striker_id", "striker_name", "runs"]
    ].head()
)

Batting dataset shape: (94672, 37)


,match_id,match_date,comp_name,striker_id,striker_name,runs
0,14094,2022-03-26,IPL 2022,6059,Ruturaj Gaikwad,0
1,14094,2022-03-26,IPL 2022,6059,Ruturaj Gaikwad,0
2,14094,2022-03-26,IPL 2022,6059,Ruturaj Gaikwad,0
3,14094,2022-03-26,IPL 2022,6059,Ruturaj Gaikwad,0
4,14094,2022-03-26,IPL 2022,6059,Ruturaj Gaikwad,0


In [4]:
print("Unique batters:", batting_data["striker_id"].nunique())

print("\nCompetition-wise deliveries:")
print(batting_data["comp_name"].value_counts())

print("\nMissing values:")
print(
    batting_data[
        ["striker_id", "striker_name", "runs",
         "extras_type", "extras_2_runs"]
    ].isnull().sum()
)

Unique batters: 480

Competition-wise deliveries:
comp_name
IPL 2022                                 17909
IPL 2023                                 17863
IPL 2026                                 17527
IPL 2025                                 17285
IPL 2024                                 17103
Mens T20 Emerging Teams Asia Cup 2024     3519
Asia Cup Rising Stars 2025                3466
Name: count, dtype: int64

Missing values:
striker_id           0
striker_name         0
runs                 0
extras_type      89077
extras_2_runs        0
dtype: int64


In [5]:
for extra in ["wide", "legbyes", "byes", "Overstep"]:
    print(f"\n--- {extra} ---")

    display(
        batting_data[
            batting_data["extras_type"] == extra
        ][
            ["striker_name", "bowler_name", "runs",
             "extras1_runs", "extras_2_runs",
             "extras_type", "is_four", "is_six"]
        ].head(5)
    )


--- wide ---


,striker_name,bowler_name,runs,extras1_runs,extras_2_runs,extras_type,is_four,is_six
2,Ruturaj Gaikwad,Umesh Yadav,0,1,0,wide,False,False
5,Robin Uthappa,Umesh Yadav,0,1,0,wide,False,False
50,Robin Uthappa,Varun Chakravarthy,0,1,0,wide,False,False
115,Ravindra Jadeja,Shivam Mavi,0,1,0,wide,False,False
264,Rohit Sharma,Kamlesh Nagarkoti,0,1,0,wide,False,False



--- legbyes ---


,striker_name,bowler_name,runs,extras1_runs,extras_2_runs,extras_type,is_four,is_six
341,Ishan Kishan,Shardul Thakur,0,1,0,legbyes,False,False
467,Lalit Yadav,Basil Thampi,0,4,0,legbyes,False,False
484,Faf du Plessis,Arshdeep Singh,0,4,0,legbyes,False,False
487,Faf du Plessis,Arshdeep Singh,0,1,0,legbyes,False,False
593,Dinesh Karthik,Odean Smith,0,1,0,legbyes,False,False



--- byes ---


,striker_name,bowler_name,runs,extras1_runs,extras_2_runs,extras_type,is_four,is_six
172,Nitish Rana,Shivam Dube,0,4,0,byes,False,False
519,Anuj Rawat,Rahul Chahar,0,4,0,byes,False,False
602,Dinesh Karthik,Sandeep Sharma,0,1,0,byes,False,False
1286,Andre Russell,Mohammed Siraj,0,2,0,byes,False,False
1357,Sherfane Rutherford,Umesh Yadav,0,4,0,byes,False,False



--- Overstep ---


,striker_name,bowler_name,runs,extras1_runs,extras_2_runs,extras_type,is_four,is_six
0,Ruturaj Gaikwad,Umesh Yadav,0,1,0,Overstep,False,False
978,Jos Buttler,Bhuvneshwar Kumar,0,1,0,Overstep,False,False
990,Yashasvi Jaiswal,Bhuvneshwar Kumar,0,1,0,Overstep,False,False
996,Jos Buttler,Umran Malik,4,1,0,Overstep,True,False
1001,Yashasvi Jaiswal,Washington Sundar,1,1,0,Overstep,False,False


In [6]:
print("Extras distribution:")
print(batting_data["extras_type"].value_counts(dropna=False))

print("\nRuns by extras category:")
print(
    batting_data.groupby(
        "extras_type", dropna=False
    )["runs"].agg(["count", "sum", "min", "max"])
)

Extras distribution:
extras_type
NaN         89077
wide         3792
legbyes      1196
Overstep      268
byes          212
other         127
Name: count, dtype: int64

Runs by extras category:
             count     sum  min  max
extras_type                         
Overstep       268     398    0    6
byes           212       0    0    0
legbyes       1196       1    0    1
other          127     210    0    6
wide          3792       0    0    0
NaN          89077  131419    0    6


In [7]:
# Investigate unusual legbye and other deliveries

print("Legbye records with runs:")
display(
    batting_data[
        (batting_data["extras_type"] == "legbyes") &
        (batting_data["runs"] > 0)
    ][
        ["match_id", "striker_name", "bowler_name",
         "runs", "extras_2_runs", "is_four", "is_six"]
    ]
)

print("\nOther extras records:")
display(
    batting_data[
        batting_data["extras_type"] == "other"
    ][
        ["striker_name", "runs", "extras_2_runs",
         "is_four", "is_six"]
    ].head(10)
)

Legbye records with runs:


,match_id,striker_name,bowler_name,runs,extras_2_runs,is_four,is_six
56241,37099,Nimesh Vimukthi,Qais Ahmad,1,0,False,False



Other extras records:


,striker_name,runs,extras_2_runs,is_four,is_six
118,Mahendra Singh Dhoni,6,0,False,True
1222,Bhuvneshwar Kumar,0,0,False,False
1821,Kagiso Rabada,0,0,False,False
2174,Kieron Pollard,0,0,False,False
3384,Patrick Cummins,2,0,False,False
3591,Quinton de Kock,6,0,False,True
3606,Deepak Hooda,0,0,False,False
4103,Rahul Tripathi,4,0,True,False
5304,Dinesh Karthik,1,0,False,False
5636,Shubman Gill,0,0,False,False


In [8]:
# Create a separate batter-runs column
batting_data["batter_runs"] = batting_data["runs"]

# Do not credit legbye runs to the batter
batting_data.loc[
    batting_data["extras_type"] == "legbyes",
    "batter_runs"
] = 0

# A wide is not a ball faced
batting_data["ball_faced"] = (
    batting_data["extras_type"] != "wide"
)

# Count dismissals credited to the striker
batting_data["dismissal_for_striker"] = (
    (batting_data["dismissed_player_id"] == batting_data["striker_id"]) &
    (batting_data["is_wicket"] == True)
)

print("Prepared batting records successfully!")

display(
    batting_data[
        ["striker_name", "runs", "batter_runs",
         "extras_type", "ball_faced",
         "dismissal_for_striker"]
    ].head(10)
)

Prepared batting records successfully!


,striker_name,runs,batter_runs,extras_type,ball_faced,dismissal_for_striker
0,Ruturaj Gaikwad,0,0,Overstep,True,False
1,Ruturaj Gaikwad,0,0,NaN,True,False
2,Ruturaj Gaikwad,0,0,wide,False,False
3,Ruturaj Gaikwad,0,0,NaN,True,False
4,Ruturaj Gaikwad,0,0,NaN,True,True
5,Robin Uthappa,0,0,wide,False,False
6,Robin Uthappa,0,0,NaN,True,False
7,Robin Uthappa,0,0,NaN,True,False
8,Robin Uthappa,0,0,NaN,True,False
9,Devon Conway,0,0,NaN,True,False


In [9]:
batting_stats = (
    batting_data.groupby(
        ["striker_id", "striker_name"]
    )
    .agg(
        runs=("batter_runs", "sum"),
        balls_faced=("ball_faced", "sum"),
        fours=("is_four", "sum"),
        sixes=("is_six", "sum"),
        dismissals=("dismissal_for_striker", "sum"),
        matches_played=("match_id", "nunique")
    )
    .reset_index()
)

# Calculate strike rate
batting_stats["strike_rate"] = (
    batting_stats["runs"] /
    batting_stats["balls_faced"].replace(0, np.nan)
) * 100

# Calculate batting average
batting_stats["batting_average"] = (
    batting_stats["runs"] /
    batting_stats["dismissals"].replace(0, np.nan)
)

# Calculate boundary runs
batting_stats["boundary_runs"] = (
    batting_stats["fours"] * 4 +
    batting_stats["sixes"] * 6
)

# Calculate boundary percentage
batting_stats["boundary_percentage"] = (
    batting_stats["boundary_runs"] /
    batting_stats["runs"].replace(0, np.nan)
) * 100

# Round calculated metrics
batting_stats = batting_stats.round(2)

print("Total batters:", len(batting_stats))

display(
    batting_stats.sort_values(
        "runs", ascending=False
    ).head(20)
)

Total batters: 480


,striker_id,striker_name,runs,balls_faced,fours,sixes,dismissals,matches_played,strike_rate,batting_average,boundary_runs,boundary_percentage
224,6288,Shubman Gill,3181,2084,309,116,68,76,152.64,46.78,1932,60.74
3,47,Virat Kohli,3053,2091,298,106,62,76,146.01,49.24,1828,59.88
33,813,Jos Buttler,2678,1791,264,121,61,72,149.53,43.90,1782,66.54
41,953,Lokesh Rahul,2548,1781,227,105,55,64,143.07,46.33,1538,60.36
363,23450,Sai Sudharsan,2515,1686,258,82,51,57,149.17,49.31,1524,60.60
264,6819,Yashasvi Jaiswal,2304,1485,276,98,63,69,155.15,36.57,1692,73.44
220,6281,Abhishek Sharma,2272,1282,223,140,70,73,177.22,32.46,1732,76.23
63,1699,Suryakumar Yadav,2240,1358,221,110,55,64,164.95,40.73,1544,68.93
154,4117,Ishan Kishan,2148,1443,227,92,66,71,148.86,32.55,1460,67.97
75,2258,Sanju Samson,2113,1385,196,111,57,69,152.56,37.07,1450,68.62


In [10]:
top_run_scorers = batting_stats.sort_values(
    by="runs",
    ascending=False
).head(20)

display(top_run_scorers)

,striker_id,striker_name,runs,balls_faced,fours,sixes,dismissals,matches_played,strike_rate,batting_average,boundary_runs,boundary_percentage
224,6288,Shubman Gill,3181,2084,309,116,68,76,152.64,46.78,1932,60.74
3,47,Virat Kohli,3053,2091,298,106,62,76,146.01,49.24,1828,59.88
33,813,Jos Buttler,2678,1791,264,121,61,72,149.53,43.90,1782,66.54
41,953,Lokesh Rahul,2548,1781,227,105,55,64,143.07,46.33,1538,60.36
363,23450,Sai Sudharsan,2515,1686,258,82,51,57,149.17,49.31,1524,60.60
264,6819,Yashasvi Jaiswal,2304,1485,276,98,63,69,155.15,36.57,1692,73.44
220,6281,Abhishek Sharma,2272,1282,223,140,70,73,177.22,32.46,1732,76.23
63,1699,Suryakumar Yadav,2240,1358,221,110,55,64,164.95,40.73,1544,68.93
154,4117,Ishan Kishan,2148,1443,227,92,66,71,148.86,32.55,1460,67.97
75,2258,Sanju Samson,2113,1385,196,111,57,69,152.56,37.07,1450,68.62


In [11]:
top_strike_rates = (
    batting_stats[
        (batting_stats["balls_faced"] >= 100) &
        (batting_stats["dismissals"] >= 1)
    ]
    .sort_values("strike_rate", ascending=False)
    .head(20)
)

display(top_strike_rates)

,striker_id,striker_name,runs,balls_faced,fours,sixes,dismissals,matches_played,strike_rate,batting_average,boundary_runs,boundary_percentage
468,50284,Vaibhav Sooryavanshi,1267,547,101,118,27,27,231.63,46.93,1112,87.77
188,4847,Finn Allen,349,163,33,28,10,11,214.11,34.90,300,85.96
284,7341,Jake McGurk,385,193,39,30,14,15,199.48,27.50,336,87.27
377,25669,Priyansh Arya,983,515,96,67,35,35,190.87,28.09,786,79.96
462,48504,Ayush Mhatre,441,240,51,23,12,13,183.75,36.75,342,77.55
338,16284,Habibur Rahman Sohan,229,126,15,21,4,5,181.75,57.25,186,81.22
228,6396,Tim David,1150,647,76,83,33,58,177.74,34.85,802,69.74
385,26843,Maaz Sadaqat,259,146,20,19,2,5,177.40,129.50,194,74.90
220,6281,Abhishek Sharma,2272,1282,223,140,70,73,177.22,32.46,1732,76.23
81,2367,Travis Head,1351,767,160,70,41,42,176.14,32.95,1060,78.46


In [12]:
ipl_data = batting_data[
    batting_data["comp_name"].str.startswith("IPL", na=False)
].copy()

print("IPL deliveries:", len(ipl_data))
print("IPL matches:", ipl_data["match_id"].nunique())
print("IPL batters:", ipl_data["striker_id"].nunique())
print("IPL seasons:", sorted(ipl_data["comp_name"].unique()))

IPL deliveries: 87687
IPL matches: 367
IPL batters: 341
IPL seasons: ['IPL 2022', 'IPL 2023', 'IPL 2024', 'IPL 2025', 'IPL 2026']


In [13]:
ipl_batting_stats = (
    ipl_data.groupby(["striker_id", "striker_name"])
    .agg(
        runs=("batter_runs", "sum"),
        balls_faced=("ball_faced", "sum"),
        fours=("is_four", "sum"),
        sixes=("is_six", "sum"),
        dismissals=("dismissal_for_striker", "sum"),
        matches_played=("match_id", "nunique")
    )
    .reset_index()
)

ipl_batting_stats["strike_rate"] = (
    ipl_batting_stats["runs"] /
    ipl_batting_stats["balls_faced"].replace(0, np.nan)
) * 100

ipl_batting_stats["batting_average"] = (
    ipl_batting_stats["runs"] /
    ipl_batting_stats["dismissals"].replace(0, np.nan)
)

ipl_batting_stats["boundary_runs"] = (
    4 * ipl_batting_stats["fours"] +
    6 * ipl_batting_stats["sixes"]
)

ipl_batting_stats["boundary_percentage"] = (
    ipl_batting_stats["boundary_runs"] /
    ipl_batting_stats["runs"].replace(0, np.nan)
) * 100

ipl_batting_stats = ipl_batting_stats.round(2)

print("IPL batting statistics created:", len(ipl_batting_stats))

IPL batting statistics created: 341


In [14]:
display(
    ipl_batting_stats.sort_values(
        "runs", ascending=False
    ).head(20)
)

,striker_id,striker_name,runs,balls_faced,fours,sixes,dismissals,matches_played,strike_rate,batting_average,boundary_runs,boundary_percentage
200,6288,Shubman Gill,3181,2084,309,116,68,76,152.64,46.78,1932,60.74
3,47,Virat Kohli,3053,2091,298,106,62,76,146.01,49.24,1828,59.88
33,813,Jos Buttler,2678,1791,264,121,61,72,149.53,43.90,1782,66.54
41,953,Lokesh Rahul,2548,1781,227,105,55,64,143.07,46.33,1538,60.36
288,23450,Sai Sudharsan,2515,1686,258,82,51,57,149.17,49.31,1524,60.60
230,6819,Yashasvi Jaiswal,2304,1485,276,98,63,69,155.15,36.57,1692,73.44
63,1699,Suryakumar Yadav,2240,1358,221,110,55,64,164.95,40.73,1544,68.93
139,4117,Ishan Kishan,2148,1443,227,92,66,71,148.86,32.55,1460,67.97
197,6281,Abhishek Sharma,2138,1216,207,132,66,69,175.82,32.39,1620,75.77
74,2258,Sanju Samson,2113,1385,196,111,57,69,152.56,37.07,1450,68.62


In [15]:
ipl_sr_leaders = (
    ipl_batting_stats[
        ipl_batting_stats["balls_faced"] >= 100
    ]
    .sort_values("strike_rate", ascending=False)
)

display(ipl_sr_leaders.head(20))

,striker_id,striker_name,runs,balls_faced,fours,sixes,dismissals,matches_played,strike_rate,batting_average,boundary_runs,boundary_percentage
333,50284,Vaibhav Sooryavanshi,1028,449,81,96,23,23,228.95,44.70,900,87.55
171,4847,Finn Allen,349,163,33,28,10,11,214.11,34.90,300,85.96
245,7341,Jake McGurk,385,193,39,30,14,15,199.48,27.50,336,87.27
294,25669,Priyansh Arya,909,471,87,63,31,31,192.99,29.32,726,79.87
330,48504,Ayush Mhatre,441,240,51,23,12,13,183.75,36.75,342,77.55
204,6396,Tim David,1150,647,76,83,33,58,177.74,34.85,802,69.74
80,2367,Travis Head,1351,767,160,70,41,42,176.14,32.95,1060,78.46
197,6281,Abhishek Sharma,2138,1216,207,132,66,69,175.82,32.39,1620,75.77
227,6726,Romario Shepherd,268,153,18,22,11,19,175.16,24.36,204,76.12
181,5553,Phil Salt,1258,721,138,67,37,40,174.48,34.00,954,75.83


In [16]:
season_batting_stats = (
    ipl_data.groupby(
        ["comp_name", "striker_id", "striker_name"]
    )
    .agg(
        runs=("batter_runs", "sum"),
        balls_faced=("ball_faced", "sum"),
        fours=("is_four", "sum"),
        sixes=("is_six", "sum"),
        dismissals=("dismissal_for_striker", "sum"),
        matches_played=("match_id", "nunique")
    )
    .reset_index()
)

season_batting_stats["strike_rate"] = (
    season_batting_stats["runs"] /
    season_batting_stats["balls_faced"].replace(0, np.nan)
) * 100

season_batting_stats["batting_average"] = (
    season_batting_stats["runs"] /
    season_batting_stats["dismissals"].replace(0, np.nan)
)

season_batting_stats = season_batting_stats.round(2)

print("Season-wise batting statistics created:",
      len(season_batting_stats))

display(
    season_batting_stats[
        season_batting_stats["striker_name"].isin([
            "Vaibhav Sooryavanshi",
            "Priyansh Arya",
            "Ayush Mhatre",
            "Prabhsimran Singh",
            "Ashutosh Sharma",
            "Abhishek Sharma"
        ])
    ].sort_values(
        ["striker_name", "comp_name"]
    )
)

Season-wise batting statistics created: 862


,comp_name,striker_id,striker_name,runs,balls_faced,fours,sixes,dismissals,matches_played,strike_rate,batting_average
123,IPL 2022,6281,Abhishek Sharma,426,320,47,13,14,14,133.12,30.43
294,IPL 2023,6281,Abhishek Sharma,226,157,28,6,11,11,143.95,20.55
458,IPL 2024,6281,Abhishek Sharma,484,237,36,42,15,16,204.22,32.27
613,IPL 2025,6281,Abhishek Sharma,439,227,46,28,12,13,193.39,36.58
771,IPL 2026,6281,Abhishek Sharma,563,275,50,43,14,15,204.73,40.21
497,IPL 2024,8101,Ashutosh Sharma,189,113,10,15,7,9,167.26,27.00
651,IPL 2025,8101,Ashutosh Sharma,204,127,14,13,7,9,160.63,29.14
807,IPL 2026,8101,Ashutosh Sharma,171,94,15,12,5,8,181.91,34.20
682,IPL 2025,48504,Ayush Mhatre,240,127,31,11,7,7,188.98,34.29
851,IPL 2026,48504,Ayush Mhatre,201,113,20,12,5,6,177.88,40.20


In [17]:
ipl_2026 = season_batting_stats[
    season_batting_stats["comp_name"] == "IPL 2026"
].copy()

print("Total IPL 2026 batters:", len(ipl_2026))

# Top run scorers
print("\nTOP 25 RUN SCORERS — IPL 2026")

display(
    ipl_2026.sort_values(
        "runs", ascending=False
    ).head(25)
)

# Top strike rates with a minimum of 100 balls
print("\nTOP STRIKE RATES — MINIMUM 100 BALLS")

display(
    ipl_2026[
        ipl_2026["balls_faced"] >= 100
    ].sort_values(
        "strike_rate", ascending=False
    ).head(25)
)

Total IPL 2026 batters: 175

TOP 25 RUN SCORERS — IPL 2026


,comp_name,striker_id,striker_name,runs,balls_faced,fours,sixes,dismissals,matches_played,strike_rate,batting_average
854,IPL 2026,50284,Vaibhav Sooryavanshi,776,327,63,72,16,16,237.31,48.50
773,IPL 2026,6288,Shubman Gill,732,449,74,33,16,16,163.03,45.75
822,IPL 2026,23450,Sai Sudharsan,722,457,75,30,16,17,157.99,45.12
687,IPL 2026,47,Virat Kohli,675,407,73,25,12,16,165.85,56.25
748,IPL 2026,4260,Heinrich Klaasen,624,390,48,31,13,15,160.00,48.00
743,IPL 2026,4117,Ishan Kishan,602,330,60,32,14,15,182.42,43.00
699,IPL 2026,953,Lokesh Rahul,593,340,56,31,13,14,174.41,45.62
771,IPL 2026,6281,Abhishek Sharma,563,275,50,43,14,15,204.73,40.21
695,IPL 2026,818,Mitchell Marsh,563,345,51,36,12,13,163.19,46.92
694,IPL 2026,813,Jos Buttler,526,345,51,26,14,17,152.46,37.57



TOP STRIKE RATES — MINIMUM 100 BALLS


,comp_name,striker_id,striker_name,runs,balls_faced,fours,sixes,dismissals,matches_played,strike_rate,batting_average
854,IPL 2026,50284,Vaibhav Sooryavanshi,776,327,63,72,16,16,237.31,48.50
760,IPL 2026,4847,Finn Allen,349,163,33,28,10,11,214.11,34.90
827,IPL 2026,25669,Priyansh Arya,364,172,27,32,13,13,211.63,28.00
771,IPL 2026,6281,Abhishek Sharma,563,275,50,43,14,15,204.73,40.21
763,IPL 2026,5039,Rajat Patidar,501,260,30,42,12,14,192.69,41.75
776,IPL 2026,6396,Tim David,305,162,22,23,9,15,188.27,33.89
741,IPL 2026,4007,Ryan Rickelton,448,240,31,38,11,12,186.67,40.73
802,IPL 2026,7544,Venkatesh Iyer,209,112,21,13,4,6,186.61,52.25
757,IPL 2026,4736,Josh Inglis,266,143,32,12,5,5,186.01,53.20
743,IPL 2026,4117,Ishan Kishan,602,330,60,32,14,15,182.42,43.00


In [18]:
playing_eleven = pd.read_csv("../data/playing_eleven.csv")
player_mapping = pd.read_csv(
    "../data/competition_team_player_mapping.csv"
)

# Get IPL 2026 competition ID
ipl_2026_comp_id = competitions.loc[
    competitions["comp_name"] == "IPL 2026",
    "comp_id"
].iloc[0]

# Get IPL 2026 matches
ipl_2026_matches = matches[
    matches["comp_id"] == ipl_2026_comp_id
][["match_id", "comp_id"]]

# Get players who actually played
ipl_2026_xi = playing_eleven.merge(
    ipl_2026_matches,
    on="match_id",
    how="inner"
)

ipl_2026_xi = ipl_2026_xi[
    ipl_2026_xi["is_played"] == True
].copy()

# Attach player and team names
ipl_2026_xi = ipl_2026_xi.merge(
    player_mapping[
        ["comp_id", "team_id", "player_id",
         "player_name", "team_name"]
    ],
    on=["comp_id", "team_id", "player_id"],
    how="left"
)

# Summarise player appearances and batting position
player_roles = (
    ipl_2026_xi.groupby(
        ["player_id", "player_name", "team_name"]
    )
    .agg(
        matches_played=("match_id", "nunique"),
        average_batting_position=("batting_order", "mean")
    )
    .reset_index()
)

player_roles["average_batting_position"] = (
    player_roles["average_batting_position"].round(1)
)

print("IPL 2026 player-role records:", len(player_roles))

display(
    player_roles.sort_values(
        ["average_batting_position", "matches_played"]
    ).head(20)
)

IPL 2026 player-role records: 202


,player_id,player_name,team_name,matches_played,average_batting_position
84,5553,Phil Salt,Royal Challengers Bengaluru,6,1.0
154,23311,Jacob Bethell,Royal Challengers Bengaluru,7,1.0
23,2258,Sanju Samson,Chennai Super Kings,14,1.0
116,6819,Yashasvi Jaiswal,Rajasthan Royals,16,1.0
155,23450,Sai Sudharsan,Gujarat Titans,17,1.0
161,25669,Priyansh Arya,Punjab Kings,13,1.1
117,6931,Pathum Nissanka,Delhi Capitals,10,1.2
22,2101,Quinton de Kock,Mumbai Indians,3,1.3
52,3700,Tim Seifert,Kolkata Knight Riders,3,1.3
57,4007,Ryan Rickelton,Mumbai Indians,12,1.3


In [19]:
# Get IPL 2026 batting statistics
ipl_2026_batting_stats = season_batting_stats[
    season_batting_stats["comp_name"] == "IPL 2026"
].copy()

# Merge batting statistics with player roles
ipl_2026_player_analysis = ipl_2026_batting_stats.merge(
    player_roles,
    left_on=["striker_id", "striker_name"],
    right_on=["player_id", "player_name"],
    how="left"
)

# Select useful columns
analysis_columns = [
    "striker_name",
    "team_name",
    "matches_played_x",
    "average_batting_position",
    "runs",
    "balls_faced",
    "batting_average",
    "strike_rate",
    "fours",
    "sixes"
]

ipl_2026_player_analysis = (
    ipl_2026_player_analysis[analysis_columns]
    .rename(columns={
        "matches_played_x": "batting_matches"
    })
)

# Top 30 by runs
display(
    ipl_2026_player_analysis
    .sort_values("runs", ascending=False)
    .head(30)
)

# High strike-rate players with at least 50 balls
display(
    ipl_2026_player_analysis[
        ipl_2026_player_analysis["balls_faced"] >= 50
    ]
    .sort_values("strike_rate", ascending=False)
    .head(30)
)

,striker_name,team_name,batting_matches,average_batting_position,runs,balls_faced,batting_average,strike_rate,fours,sixes
167,Vaibhav Sooryavanshi,Rajasthan Royals,16,2.0,776,327,48.50,237.31,63,72
86,Shubman Gill,Gujarat Titans,16,2.0,732,449,45.75,163.03,74,33
135,Sai Sudharsan,Gujarat Titans,17,1.0,722,457,45.12,157.99,75,30
0,Virat Kohli,Royal Challengers Bengaluru,16,2.0,675,407,56.25,165.85,73,25
61,Heinrich Klaasen,Sunrisers Hyderabad,15,4.3,624,390,48.00,160.00,48,31
56,Ishan Kishan,Sunrisers Hyderabad,15,3.0,602,330,43.00,182.42,60,32
12,Lokesh Rahul,Delhi Capitals,14,1.8,593,340,45.62,174.41,56,31
84,Abhishek Sharma,Sunrisers Hyderabad,15,1.3,563,275,40.21,204.73,50,43
8,Mitchell Marsh,Lucknow Super Giants,13,1.3,563,345,46.92,163.19,51,36
7,Jos Buttler,Gujarat Titans,17,3.1,526,345,37.57,152.46,51,26


,striker_name,team_name,batting_matches,average_batting_position,runs,balls_faced,batting_average,strike_rate,fours,sixes
167,Vaibhav Sooryavanshi,Rajasthan Royals,16,2.0,776,327,48.50,237.31,63,72
73,Finn Allen,Kolkata Knight Riders,11,1.9,349,163,34.90,214.11,33,28
140,Priyansh Arya,Punjab Kings,13,1.1,364,172,28.00,211.63,27,32
84,Abhishek Sharma,Sunrisers Hyderabad,15,1.3,563,275,40.21,204.73,50,43
91,Urvil Patel,Chennai Super Kings,7,3.1,129,64,18.43,201.56,6,14
76,Rajat Patidar,Royal Challengers Bengaluru,14,3.9,501,260,41.75,192.69,30,42
32,Shashank Singh,Punjab Kings,9,6.2,132,70,22.00,188.57,11,8
89,Tim David,Royal Challengers Bengaluru,15,5.8,305,162,33.89,188.27,22,23
54,Ryan Rickelton,Mumbai Indians,12,1.3,448,240,40.73,186.67,31,38
115,Venkatesh Iyer,Royal Challengers Bengaluru,6,4.7,209,112,52.25,186.61,21,13


In [20]:
# Keep players with a reasonable sample size
batting_candidate_pool_2026 = ipl_2026_player_analysis[
    (ipl_2026_player_analysis["batting_matches"] >= 5) &
    (ipl_2026_player_analysis["balls_faced"] >= 100)
].copy()

# Classify players by their average batting position
def classify_batting_role(position):
    if position <= 2.5:
        return "Top Order"
    elif position <= 4.5:
        return "Middle Order"
    else:
        return "Finisher / Lower Order"

batting_candidate_pool_2026["batting_role"] = (
    batting_candidate_pool_2026["average_batting_position"]
    .apply(classify_batting_role)
)

# Sort by runs to inspect the full candidate pool
batting_candidate_pool_2026 = (
    batting_candidate_pool_2026
    .sort_values(
        ["runs", "strike_rate"],
        ascending=False
    )
    .reset_index(drop=True)
)

display(batting_candidate_pool_2026)

print("Total preliminary candidates:",
      len(batting_candidate_pool_2026))

print("\nCandidates by batting role:")
print(batting_candidate_pool_2026["batting_role"].value_counts())

,striker_name,team_name,batting_matches,average_batting_position,runs,balls_faced,batting_average,strike_rate,fours,sixes,batting_role
0,Vaibhav Sooryavanshi,Rajasthan Royals,16,2.0,776,327,48.50,237.31,63,72,Top Order
1,Shubman Gill,Gujarat Titans,16,2.0,732,449,45.75,163.03,74,33,Top Order
2,Sai Sudharsan,Gujarat Titans,17,1.0,722,457,45.12,157.99,75,30,Top Order
3,Virat Kohli,Royal Challengers Bengaluru,16,2.0,675,407,56.25,165.85,73,25,Top Order
4,Heinrich Klaasen,Sunrisers Hyderabad,15,4.3,624,390,48.00,160.00,48,31,Middle Order
5,Ishan Kishan,Sunrisers Hyderabad,15,3.0,602,330,43.00,182.42,60,32,Middle Order
6,Lokesh Rahul,Delhi Capitals,14,1.8,593,340,45.62,174.41,56,31,Top Order
7,Abhishek Sharma,Sunrisers Hyderabad,15,1.3,563,275,40.21,204.73,50,43,Top Order
8,Mitchell Marsh,Lucknow Super Giants,13,1.3,563,345,46.92,163.19,51,36,Top Order
9,Jos Buttler,Gujarat Titans,17,3.1,526,345,37.57,152.46,51,26,Middle Order


Total preliminary candidates: 60

Candidates by batting role:
batting_role
Top Order                 20
Middle Order              20
Finisher / Lower Order    20
Name: count, dtype: int64


In [21]:
print(ipl_2026_player_analysis.columns.tolist())

['striker_name', 'team_name', 'batting_matches', 'average_batting_position', 'runs', 'balls_faced', 'batting_average', 'strike_rate', 'fours', 'sixes']


In [22]:

# Select IPL 2025 and IPL 2026
recent_ipl_data = season_batting_stats[
    season_batting_stats["comp_name"].isin(
        ["IPL 2025", "IPL 2026"]
    )
].copy()

# Combine both seasons for each batter
two_season_batting = (
    recent_ipl_data
    .groupby(["striker_id", "striker_name"])
    .agg(
        total_runs=("runs", "sum"),
        total_balls=("balls_faced", "sum"),
        total_dismissals=("dismissals", "sum"),
        total_matches=("matches_played", "sum"),
        total_fours=("fours", "sum"),
        total_sixes=("sixes", "sum")
    )
    .reset_index()
)

# Calculate combined performance
two_season_batting["batting_average"] = (
    two_season_batting["total_runs"] /
    two_season_batting["total_dismissals"].replace(0, np.nan)
)

two_season_batting["strike_rate"] = (
    two_season_batting["total_runs"] /
    two_season_batting["total_balls"].replace(0, np.nan)
) * 100

two_season_batting = two_season_batting.round(2)

# Add IPL 2026 team and batting position
two_season_batting = two_season_batting.merge(
    ipl_2026_player_analysis[
        [
            "striker_name",
            "team_name",
            "average_batting_position"
        ]
    ].drop_duplicates(subset=["striker_name"]),
    on="striker_name",
    how="left",
    validate="one_to_one"
)

# Display players with at least 150 balls across these seasons
two_season_batting = two_season_batting[
    two_season_batting["total_balls"] >= 150
].copy()

display(
    two_season_batting
    .sort_values("total_runs", ascending=False)
    .head(40)
)

print("Players with at least 150 balls:", len(two_season_batting))

,striker_id,striker_name,total_runs,total_balls,total_dismissals,total_matches,total_fours,total_sixes,batting_average,strike_rate,team_name,average_batting_position
176,23450,Sai Sudharsan,1481,943,30,32,163,51,49.37,157.05,Gujarat Titans,1.0
112,6288,Shubman Gill,1382,866,28,31,136,57,49.36,159.58,Gujarat Titans,2.0
1,47,Virat Kohli,1332,861,24,31,139,44,55.50,154.70,Royal Challengers Bengaluru,2.0
12,818,Mitchell Marsh,1190,728,25,26,107,73,47.60,163.46,Lucknow Super Giants,1.3
17,953,Lokesh Rahul,1139,703,22,27,109,52,51.77,162.02,Delhi Capitals,1.8
77,4260,Heinrich Klaasen,1111,672,23,28,90,56,48.30,165.33,Sunrisers Hyderabad,4.3
135,7216,Prabhsimran Singh,1109,672,29,31,118,53,38.24,165.03,Punjab Kings,1.9
60,3647,Shreyas Iyer,1102,640,21,30,82,69,52.48,172.19,Punjab Kings,3.7
11,813,Jos Buttler,1064,675,23,30,103,50,46.26,157.63,Gujarat Titans,3.1
214,50284,Vaibhav Sooryavanshi,1028,449,23,23,81,96,44.70,228.95,Rajasthan Royals,2.0


Players with at least 150 balls: 71


In [23]:
# Keep IPL 2025 and IPL 2026
recent_seasons = season_batting_stats[
    season_batting_stats["comp_name"].isin(
        ["IPL 2025", "IPL 2026"]
    )
].copy()

# Create a player-by-season comparison
season_comparison = recent_seasons.pivot_table(
    index=["striker_id", "striker_name"],
    columns="comp_name",
    values=[
        "runs",
        "strike_rate",
        "batting_average",
        "matches_played"
    ],
    aggfunc="first"
)

# Flatten the column names
season_comparison.columns = [
    f"{metric}_{season.replace('IPL ', '')}"
    for metric, season in season_comparison.columns
]

season_comparison = season_comparison.reset_index()

# Add 2026 team and batting position
season_comparison = season_comparison.merge(
    ipl_2026_player_analysis[
        [
            "striker_name",
            "team_name",
            "average_batting_position"
        ]
    ].drop_duplicates(subset=["striker_name"]),
    on="striker_name",
    how="left",
    validate="one_to_one"
)

# Show players who appeared in both seasons
consistent_players = season_comparison.dropna(
    subset=["runs_2025", "runs_2026"]
).copy()

consistent_players = consistent_players.sort_values(
    "runs_2026",
    ascending=False
)

display(consistent_players.head(40))

print("Players appearing in both seasons:",
      len(consistent_players))

,striker_id,striker_name,batting_average_2025,batting_average_2026,matches_played_2025,matches_played_2026,runs_2025,runs_2026,strike_rate_2025,strike_rate_2026,team_name,average_batting_position
214,50284,Vaibhav Sooryavanshi,36.00,48.50,7.0,16.0,252.0,776.0,206.56,237.31,Rajasthan Royals,2.0
112,6288,Shubman Gill,54.17,45.75,15.0,16.0,650.0,732.0,155.88,163.03,Gujarat Titans,2.0
176,23450,Sai Sudharsan,54.21,45.12,15.0,17.0,759.0,722.0,156.17,157.99,Gujarat Titans,1.0
1,47,Virat Kohli,54.75,56.25,15.0,16.0,657.0,675.0,144.71,165.85,Royal Challengers Bengaluru,2.0
77,4260,Heinrich Klaasen,48.70,48.00,13.0,15.0,487.0,624.0,172.70,160.00,Sunrisers Hyderabad,4.3
72,4117,Ishan Kishan,35.40,43.00,13.0,15.0,354.0,602.0,152.59,182.42,Sunrisers Hyderabad,3.0
17,953,Lokesh Rahul,60.67,45.62,13.0,14.0,546.0,593.0,150.41,174.41,Delhi Capitals,1.8
110,6281,Abhishek Sharma,36.58,40.21,13.0,15.0,439.0,563.0,193.39,204.73,Sunrisers Hyderabad,1.3
12,818,Mitchell Marsh,48.23,46.92,13.0,13.0,627.0,563.0,163.71,163.19,Lucknow Super Giants,1.3
11,813,Jos Buttler,59.78,37.57,13.0,17.0,538.0,526.0,163.03,152.46,Gujarat Titans,3.1


Players appearing in both seasons: 119
